## [B] Calculating Model Parameters

In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
path = "data/processed_data/"
file_name = "01_prepared_raw_data.xlsx"
values_to_replace = [-1, 0, 1]
beta_rolling_window = 12

constituent_cut_off_date = "2016-06-01"
start_date = "2015-06-01" # 1-year lookback before cut-off date for rolling windows (Beta, Price & ESG Momentum, Dividend-Yield)
start_date_esg = "2014-12-01" 
end_date = "2025-09-30"

date_column = "date"
stock_column = "stock"

In [3]:
# Set path
os.chdir("..")
print(os.getcwd())

/workspaces/master-thesis-submission/src


In [4]:
constituents_file = pd.read_excel(path + file_name, sheet_name="constituents")
presence_matrix_file = pd.read_excel(path + file_name, sheet_name="presence_matrix")

In [5]:
price_close_stocks_file = pd.read_excel(path + file_name, sheet_name="price_close_stocks")
price_close_index_file = pd.read_excel(path + file_name, sheet_name="price_close_index")

In [6]:
free_float_mcap_file = pd.read_excel(path + file_name, sheet_name="free_float_mcap")
outstanding_mcap_file = pd.read_excel(path + file_name, sheet_name="outstanding_mcap")
outstanding_shares_file = pd.read_excel(path + file_name, sheet_name="outstanding_shares")

In [7]:
book_to_price_file = pd.read_excel(path + file_name, sheet_name="book_to_price_value_per_share")

adjusted_gross_dividend_amount_file = pd.read_excel(path + file_name, sheet_name="adjusted_gross_dividend_amount")

operating_income_ttm_file = pd.read_excel(path + file_name, sheet_name="operating_income_ttm")
total_equity_ttm_file = pd.read_excel(path + file_name, sheet_name="total_equity_ttm")
total_assets_reported_ttm_file = pd.read_excel(path + file_name, sheet_name="total_assets_reported_ttm")

In [8]:
esg_score_file = pd.read_excel(path + file_name, sheet_name="esg_score")

industry_file = pd.read_excel(path + file_name, sheet_name="industry")
currency_file = pd.read_excel(path + file_name, sheet_name="currency")
country_of_headquarters_file = pd.read_excel(path + file_name, sheet_name="country_of_headquarters")

recession_indicator = pd.read_excel(path + file_name, sheet_name="recession_indicator")

In [9]:
fama_french_data = pd.read_excel(path + file_name, sheet_name="fama_french_data")
bond_yields_file = pd.read_excel(path + file_name, sheet_name="bond_yields")
gov_bond_yields = pd.read_excel(path + file_name, sheet_name="gov_bond_yields")
risk_free_rate_file = pd.read_excel(path + file_name, sheet_name="risk_free_rate")

## 1. Global Helper Functions

In [10]:
def replaceValuesWithNaN(df, values_to_replace, date_column="date"):
    """
    Replace specified values in the DataFrame with NaN.
    This function is used to clean the data by replacing invalid or unwanted values.
    """
    cols = df.columns.tolist()

    if date_column in cols:
        cols = [c for c in cols if c != date_column]
        
    # Create a boolean mask where values are zero for each column
    mask = df[cols].isin(values_to_replace)

    # Replace zeros with NaN
    total_replacements = int(mask.to_numpy().sum())
    per_column_counts = mask.sum().sort_values(ascending=False)

    replacements = (
        mask.stack()
            .loc[lambda s: s]
            .reset_index()
            .rename(columns={"level_0": "date", "level_1": "column", 0: "matched"})
            .drop(columns=["matched"])
    )

    df.loc[:, cols] = df.loc[:, cols].mask(mask, np.nan)

    print(f"Replaced {total_replacements} values ({values_to_replace}) with NaN.\n")
    print("Replacements per column (only columns with >=1 replacement):")
    print(per_column_counts[per_column_counts > 0].to_string(), "\n")

    return df

In [11]:
def fill_forward_data(file):
    """
    Fills forward data to handle missing values.
    """

    df = file.copy()

    # Get list of stock columns
    stock_columns = [column for column in df.columns if column != date_column]    

    # Fill forward missing values (The most recent available data is used to fill missing values)
    df[stock_columns] = df[stock_columns].ffill()

    return df

In [12]:
def prep_financial_data(file):
    """
    Prepares financial data by subsetting date range, setting index, sorting, dropping NaNs, and replacing specified values with NaN.
    """

    df = file.copy()

    # Only include data within the specified date range
    df = df[(df[date_column] >= start_date) & (df[date_column] <= end_date) ].reset_index(drop=True)

    # Set date as index and sort
    df = df.sort_values(date_column).set_index(date_column)
 
    # Drop rows with all NaN values
    df = df.dropna(how="all")

    # Replace specified values with NaN
    df = replaceValuesWithNaN(df, values_to_replace=values_to_replace)

    # Fill forward missing values
    df = fill_forward_data(df)

    return df


## 2. Calculate Parameters

### 2.1 Price Close / Stock Return

In [ ]:
# Only include data within the specified date range
price_close_stocks_daily = price_close_stocks_file[(price_close_stocks_file[date_column] >= start_date) & (price_close_stocks_file[date_column] <= end_date)].reset_index(drop=True)

# Set date as index because resampling requires a datetime index and sort 
price_close_stocks_daily = price_close_stocks_daily.sort_values(date_column).set_index(date_column)

# Resample to quarterly and monthly frequency, taking the last available price in each quarter / month
price_close_stocks_quarterly = price_close_stocks_daily.resample("QE").last()
price_close_stocks_monthly = price_close_stocks_daily.resample("ME").last()

# Calculate quarterly / monthly stock returns and drop rows with all NaN values
stock_returns_quarterly = price_close_stocks_quarterly.pct_change(fill_method=None).dropna(how="all")
stock_returns_monthly = price_close_stocks_monthly.pct_change(fill_method=None).dropna(how="all")

# Reset index to have date as a column again
stock_returns_quarterly = stock_returns_quarterly.reset_index().rename(columns={"index": date_column})
stock_returns_monthly = stock_returns_monthly.reset_index().rename(columns={"index": date_column})
price_close_stocks_daily = price_close_stocks_daily.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
stock_returns_quarterly[date_column] = stock_returns_quarterly[date_column].dt.date
stock_returns_monthly[date_column] = stock_returns_monthly[date_column].dt.date
price_close_stocks_daily[date_column] = price_close_stocks_daily[date_column].dt.date

In [14]:
stock_returns_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-09-30,NaN,0.187898,0.037383,-0.266343,NaN,0.089905,-0.424415,-0.006757,-0.157454,...,-0.155878,-0.094283,-0.237099,-0.011851,0.135472,0.325412,-0.110236,-0.298116,-0.011259,-0.194808
1,2015-12-31,NaN,-0.177226,0.129730,0.110896,NaN,0.175786,-0.455002,0.201436,0.049931,...,0.188328,-0.054712,0.118238,0.229730,-0.123502,0.034253,0.292035,0.072213,0.115942,0.087474
2,2016-03-31,-0.020068,-0.077897,-0.088517,-0.215836,NaN,0.023026,0.714399,-0.040893,0.030407,...,-0.056465,-0.163314,-0.169574,-0.207418,-0.081070,-0.107828,-0.174658,-0.198635,-0.051020,-0.146621
3,2016-06-30,0.211924,-0.044528,0.030621,-0.140045,NaN,-0.085994,0.250263,-0.112824,0.029602,...,-0.080966,-0.162618,-0.142458,-0.178163,-0.241204,-0.113005,0.058921,0.198751,-0.177908,0.081898
4,2016-09-30,0.317562,0.254791,0.067063,0.188100,NaN,0.030177,0.282543,0.121996,0.132308,...,0.096306,0.080486,0.041830,0.566849,-0.000484,-0.176585,0.010972,0.026765,0.068864,0.036604


In [15]:
stock_returns_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-07-31,NaN,-0.037630,0.085981,-0.007379,NaN,0.158505,-0.109648,0.072823,-0.015720,...,-0.004760,0.057855,-0.031775,0.036221,0.269772,0.342771,-0.007874,-0.070205,0.022518,0.015234
1,2015-08-31,NaN,0.047545,-0.058520,-0.117672,NaN,-0.041282,-0.115514,-0.030441,-0.069262,...,-0.117169,-0.105041,-0.121169,-0.059278,-0.061062,-0.114421,-0.117460,-0.001657,-0.046046,-0.116504
2,2015-09-30,NaN,0.178323,0.014625,-0.162317,NaN,-0.018705,-0.269102,-0.045110,-0.080297,...,-0.039274,-0.043328,-0.103425,0.013699,-0.047612,0.114606,0.016187,-0.243866,0.013641,-0.102306
3,2015-10-31,NaN,-0.012629,0.123423,0.012002,NaN,0.128442,0.027673,0.117347,0.083976,...,0.095574,0.101879,0.158029,0.076182,-0.057795,0.034430,0.176991,0.122225,0.159420,0.092747
4,2015-11-30,0.240757,-0.097207,0.066560,-0.008757,NaN,0.084448,-0.239987,0.056993,0.045773,...,0.094699,-0.068993,0.060036,0.007691,-0.001450,-0.006792,0.054135,0.114348,0.022321,0.036959


### 2.2 Price Close / Index Return

In [16]:
# Only include data within the specified date range
price_close_index_daily = price_close_index_file[(price_close_index_file[date_column] >= start_date) & (price_close_index_file[date_column] <= end_date)].reset_index(drop=True)

# Set date as index and sort
price_close_index_daily = price_close_index_daily.sort_values(date_column).set_index(date_column)

# Resample to quarterly / monthly frequency, taking the last available price in each quarter / month
price_close_index_quarterly = price_close_index_daily.resample("QE").last()
price_close_index_monthly = price_close_index_daily.resample("ME").last()

# Calculate quarterly / monthly index returns and drop rows with all NaN values
index_returns_quarterly = price_close_index_quarterly.pct_change(fill_method=None).dropna(how="all")
index_returns_monthly = price_close_index_monthly.pct_change(fill_method=None).dropna(how="all")

# Reset index to have date as a column again
index_returns_quarterly = index_returns_quarterly.reset_index().rename(columns={"index": date_column})
index_returns_monthly = index_returns_monthly.reset_index().rename(columns={"index": date_column})
price_close_index_daily = price_close_index_daily.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
index_returns_quarterly[date_column] = index_returns_quarterly[date_column].dt.date
index_returns_monthly[date_column] = index_returns_monthly[date_column].dt.date
price_close_index_daily[date_column] = price_close_index_daily[date_column].dt.date

In [17]:
index_returns_quarterly.head()

,date,.STOXX
0,2015-09-30,-0.087960
1,2015-12-31,0.051873
2,2016-03-31,-0.077281
3,2016-06-30,-0.022694
4,2016-09-30,0.039530


In [18]:
index_returns_monthly.head()

,date,.STOXX
0,2015-07-31,0.039495
1,2015-08-31,-0.084719
2,2015-09-30,-0.041401
3,2015-10-31,0.079650
4,2015-11-30,0.026527


### 2.3 Size (Market Capitalization)

#### 2.3.1 Free Float MCAP

In [19]:
# Only include data within the specified date range
free_float_mcap_file_subsetted = free_float_mcap_file[(free_float_mcap_file[date_column] >= start_date) & (free_float_mcap_file[date_column] <= end_date)].reset_index(drop=True)

# Set date as index and sort
free_float_mcap_file_subsetted = free_float_mcap_file_subsetted.sort_values(date_column).set_index(date_column)

# Resample to quarterly frequency, taking the last available market cap in each quarter
free_float_mcap_quarterly = free_float_mcap_file_subsetted.resample("QE").last()
free_float_mcap_monthly = free_float_mcap_file_subsetted.resample("ME").last()

# Drop rows with all NaN values
free_float_mcap_quarterly = free_float_mcap_quarterly.dropna(how="all")
free_float_mcap_monthly = free_float_mcap_monthly.dropna(how="all")

# Replace specified values with NaN
free_float_mcap_quarterly = replaceValuesWithNaN(free_float_mcap_quarterly, values_to_replace=values_to_replace)
free_float_mcap_monthly = replaceValuesWithNaN(free_float_mcap_monthly, values_to_replace=values_to_replace)

# Reset index to have date as a column again
free_float_mcap_quarterly = free_float_mcap_quarterly.reset_index().rename(columns={"index": date_column})
free_float_mcap_monthly = free_float_mcap_monthly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
free_float_mcap_quarterly[date_column] = free_float_mcap_quarterly[date_column].dt.date
free_float_mcap_monthly[date_column] = free_float_mcap_monthly[date_column].dt.date
free_float_mcap_file[date_column] = free_float_mcap_file[date_column].dt.date

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 



In [20]:
free_float_mcap_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,1.604204e+09,1.581934e+09,3.148755e+09,NaN,1.497385e+09,1.693444e+10,2.554939e+09,4.206480e+10,...,7.401159e+09,1.243402e+10,8.153421e+09,2.825014e+09,NaN,3.075640e+08,5.489125e+08,5.563935e+09,1.702071e+09,4.075116e+10
1,2015-07-31,NaN,1.502104e+09,1.717951e+09,3.125519e+09,NaN,1.735037e+09,1.507887e+10,2.740997e+09,4.109391e+10,...,7.365932e+09,1.315559e+10,7.927404e+09,3.450208e+09,5.328562e+07,4.129881e+08,5.445904e+08,5.173317e+09,1.740398e+09,4.137224e+10
2,2015-08-31,NaN,1.573521e+09,1.617417e+09,2.757734e+09,NaN,1.663411e+09,1.333711e+10,2.657559e+09,3.824765e+10,...,6.461400e+09,1.178378e+10,6.966847e+09,3.245685e+09,4.268157e+08,3.657337e+08,4.806226e+08,5.164742e+09,1.660260e+09,3.655225e+10
3,2015-09-30,NaN,1.854117e+09,1.641072e+09,2.317731e+09,NaN,1.632341e+09,9.722820e+09,2.537676e+09,3.474866e+10,...,6.207634e+09,1.127548e+10,6.261091e+09,3.410399e+09,4.064940e+08,4.181340e+08,4.884025e+08,3.862162e+09,1.682907e+09,3.281859e+10
4,2015-10-31,1.749688e+09,1.830702e+09,1.843619e+09,2.347563e+09,NaN,1.842003e+09,9.986387e+09,2.835464e+09,3.766673e+10,...,6.800921e+09,1.242427e+10,7.250522e+09,3.670212e+09,3.830007e+08,4.325302e+08,5.750183e+08,4.334215e+09,1.951197e+09,3.586244e+10


In [21]:
free_float_mcap_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,1.604204e+09,1.581934e+09,3.148755e+09,NaN,1.497385e+09,1.693444e+10,2.554939e+09,4.206480e+10,...,7.401159e+09,1.243402e+10,8.153421e+09,2.825014e+09,NaN,3.075640e+08,5.489125e+08,5.563935e+09,1.702071e+09,4.075116e+10
1,2015-09-30,NaN,1.854117e+09,1.641072e+09,2.317731e+09,NaN,1.632341e+09,9.722820e+09,2.537676e+09,3.474866e+10,...,6.207634e+09,1.127548e+10,6.261091e+09,3.410399e+09,4.064940e+08,4.181340e+08,4.884025e+08,3.862162e+09,1.682907e+09,3.281859e+10
2,2015-12-31,2.102187e+09,1.532970e+09,1.930577e+09,2.576969e+09,NaN,1.927645e+09,5.297176e+09,3.048856e+09,3.685222e+10,...,7.356584e+09,1.073034e+10,7.001390e+09,4.193869e+09,3.490632e+08,4.385652e+08,7.656225e+08,4.144258e+09,1.878027e+09,3.569076e+10
3,2016-03-31,2.060000e+09,1.413932e+09,1.759689e+09,2.021601e+09,NaN,1.982822e+09,9.081217e+09,2.924178e+09,3.708667e+10,...,6.932015e+09,8.989299e+09,5.774707e+09,3.284833e+09,3.149989e+08,3.912757e+08,6.352097e+08,3.266653e+09,1.871222e+09,3.046432e+10
4,2016-06-30,2.496562e+09,1.350222e+09,1.772342e+09,1.743139e+09,NaN,1.812311e+09,1.136357e+10,2.594261e+09,3.640779e+10,...,6.302067e+09,7.557542e+09,4.932859e+09,2.701593e+09,2.433948e+08,3.499752e+08,7.260737e+08,3.915903e+09,1.473550e+09,3.298225e+10


#### 2.3.2 Outstanding MCAP

In [22]:
# Only include data within the specified date range
outstanding_mcap_daily = outstanding_mcap_file[(outstanding_mcap_file[date_column] >= start_date) & (outstanding_mcap_file[date_column] <= end_date)].reset_index(drop=True)

# Set date as index and sort
outstanding_mcap_daily = outstanding_mcap_daily.sort_values(date_column).set_index(date_column)

# Resample to quarterly frequency, taking the last available market cap in each quarter
outstanding_mcap_quarterly = outstanding_mcap_daily.resample("QE").last()
outstanding_mcap_monthly = outstanding_mcap_daily.resample("ME").last()
# Drop rows with all NaN values
outstanding_mcap_quarterly = outstanding_mcap_quarterly.dropna(how="all")
outstanding_mcap_monthly = outstanding_mcap_monthly.dropna(how="all")

# Replace specified values with NaN
outstanding_mcap_quarterly = replaceValuesWithNaN(outstanding_mcap_quarterly, values_to_replace=values_to_replace)
outstanding_mcap_monthly = replaceValuesWithNaN(outstanding_mcap_monthly, values_to_replace=values_to_replace)

# Reset index to have date as a column again
outstanding_mcap_quarterly = outstanding_mcap_quarterly.reset_index().rename(columns={"index": date_column})
outstanding_mcap_monthly = outstanding_mcap_monthly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
outstanding_mcap_quarterly[date_column] = outstanding_mcap_quarterly[date_column].dt.date
outstanding_mcap_monthly[date_column] = outstanding_mcap_monthly[date_column].dt.date

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 



In [23]:
outstanding_mcap_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,2.190312e+09,3.323407e+09,3.168097e+09,NaN,2.246092e+09,1.816036e+10,2.945854e+09,4.175023e+10,...,8.288072e+09,1.267831e+10,1.286586e+10,7.387476e+09,4.619121e+07,3.401073e+08,8.877526e+08,8.031207e+09,4.419537e+09,4.075116e+10
1,2015-07-31,NaN,2.107891e+09,3.609158e+09,3.144719e+09,NaN,2.602109e+09,1.616914e+10,3.160379e+09,4.109391e+10,...,8.248624e+09,1.346487e+10,1.245705e+10,7.655058e+09,5.865230e+07,4.566863e+08,8.807624e+08,7.467373e+09,4.519056e+09,4.137224e+10
2,2015-08-31,NaN,2.208111e+09,3.397951e+09,2.774675e+09,NaN,2.494689e+09,1.430139e+10,3.064175e+09,3.824765e+10,...,7.282143e+09,1.205099e+10,1.094764e+10,7.201279e+09,4.318547e+08,4.044320e+08,7.773078e+08,7.492733e+09,4.310972e+09,3.655787e+10
3,2015-09-30,NaN,2.601868e+09,3.447646e+09,2.332478e+09,NaN,2.448317e+09,1.045289e+10,2.925949e+09,3.509965e+10,...,6.996143e+09,1.152888e+10,9.815386e+09,7.299926e+09,4.112930e+08,4.583078e+08,7.898901e+08,5.665507e+09,4.369778e+09,3.281859e+10
4,2015-10-31,5.668988e+09,2.569010e+09,3.873167e+09,2.360472e+09,NaN,2.762784e+09,1.074215e+10,3.269301e+09,3.804720e+10,...,7.664790e+09,1.270347e+10,1.136650e+10,7.856052e+09,3.875224e+08,4.740871e+08,9.298666e+08,6.357973e+09,5.066409e+09,3.586324e+10


In [24]:
outstanding_mcap_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,2.190312e+09,3.323407e+09,3.168097e+09,NaN,2.246092e+09,1.816036e+10,2.945854e+09,4.175023e+10,...,8.288072e+09,1.267831e+10,1.286586e+10,7.387476e+09,4.619121e+07,3.401073e+08,8.877526e+08,8.031207e+09,4.419537e+09,4.075116e+10
1,2015-09-30,NaN,2.601868e+09,3.447646e+09,2.332478e+09,NaN,2.448317e+09,1.045289e+10,2.925949e+09,3.509965e+10,...,6.996143e+09,1.152888e+10,9.815386e+09,7.299926e+09,4.112930e+08,4.583078e+08,7.898901e+08,5.665507e+09,4.369778e+09,3.281859e+10
2,2015-12-31,6.811088e+09,2.140750e+09,3.894909e+09,2.591139e+09,NaN,2.878698e+09,5.696876e+09,3.515341e+09,3.685222e+10,...,8.313714e+09,1.089824e+10,1.097594e+10,8.976936e+09,3.604976e+08,4.831760e+08,1.020756e+09,6.074628e+09,4.876419e+09,3.569076e+10
3,2016-03-31,6.674400e+09,1.973992e+09,3.510139e+09,2.032715e+09,NaN,2.953575e+09,9.769318e+09,3.371587e+09,3.708667e+10,...,7.844280e+09,9.126849e+09,9.075276e+09,7.126225e+09,3.253174e+08,4.310762e+08,8.429194e+08,4.878117e+09,4.627622e+09,3.046484e+10
4,2016-06-30,8.088862e+09,1.886095e+09,3.617623e+09,1.749150e+09,NaN,2.699584e+09,1.221423e+10,2.991192e+09,3.818451e+10,...,7.209163e+09,7.644490e+09,7.763235e+09,5.858680e+09,2.513678e+08,3.852781e+08,8.925852e+08,5.847646e+09,3.804331e+09,3.298225e+10


#### 2.3.3 Outstanding Shares

In [25]:
# Only include data within the specified date range
outstanding_shares_file_subsetted = outstanding_shares_file[(outstanding_shares_file[date_column] >= start_date) & (outstanding_shares_file[date_column] <= end_date) ].reset_index(drop=True)

# Set date as index and sort
outstanding_shares_file_subsetted = outstanding_shares_file_subsetted.sort_values(date_column).set_index(date_column)

# Resample to quarterly frequency, taking the last available market cap in each quarter
outstanding_shares_quarterly = outstanding_shares_file_subsetted.resample("QE").last()

# Drop rows with all NaN values
outstanding_shares_quarterly = outstanding_shares_quarterly.dropna(how="all")

# Replace specified values with NaN
outstanding_shares_quarterly = replaceValuesWithNaN(outstanding_shares_quarterly, values_to_replace=values_to_replace)

# Reset index to have date as a column again
outstanding_shares_quarterly = outstanding_shares_quarterly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
outstanding_shares_quarterly[date_column] = outstanding_shares_quarterly[date_column].dt.date

outstanding_shares_quarterly.head()

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 



,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,54764649.0,3.105988e+09,605937502.0,NaN,42160000.0,1.401836e+09,110580102.0,2.220685e+09,...,197241130.0,181743560.0,275083369.0,246619133.0,25000000.0,22953885.0,6990178.0,275041345.0,452357983.0,149123400.0
1,2015-09-30,NaN,54764649.0,3.105988e+09,608070268.0,NaN,42165000.0,1.401844e+09,110580102.0,2.215834e+09,...,197241130.0,182470367.0,275083369.0,246619133.0,196044960.0,23337075.0,6990178.0,276433595.0,452357983.0,149151048.0
2,2015-12-31,202500000.0,54764649.0,3.105988e+09,608070268.0,NaN,42165000.0,1.401862e+09,110580102.0,2.215834e+09,...,197241130.0,182472605.0,275083369.0,246619133.0,196044960.0,23788546.0,6991478.0,276433595.0,452357983.0,149156822.0
3,2016-03-31,202500000.0,54764649.0,3.070988e+09,608320662.0,NaN,42288000.0,1.402234e+09,110580102.0,2.164125e+09,...,197241130.0,182641653.0,273893369.0,247009518.0,196044960.0,23788546.0,6995182.0,277008321.0,452357983.0,149191538.0
4,2016-06-30,202500000.0,54764649.0,3.070988e+09,608706065.0,NaN,42288000.0,1.402237e+09,110580102.0,2.164125e+09,...,197241130.0,182685498.0,273217830.0,247097408.0,196044960.0,23969952.0,6995182.0,277008321.0,452357983.0,149292886.0


#### 2.3.4 Size Calculation

In [26]:
size_file = outstanding_mcap_quarterly.copy()

# Set date as index and sort
size_file = size_file.sort_values(date_column).set_index(date_column)

# Apply natural logarithm to outstanding market capitalization
for stock in size_file.columns:
    size_file[stock] = np.log(size_file[stock])

# Reset index to have date as a column again
size_file = size_file.reset_index().rename(columns={"index": date_column})

In [27]:
size_file.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,21.507310,21.924256,21.876397,NaN,21.532458,23.622507,21.803665,24.454971,...,22.838083,23.263158,23.277843,22.723052,17.648300,19.644772,20.604204,22.806601,22.209301,24.430750
1,2015-09-30,NaN,21.679496,21.960958,21.570197,NaN,21.618667,23.070144,21.796885,24.281457,...,22.668625,23.168121,23.007217,22.711130,19.834816,19.943051,20.487404,22.457662,22.197978,24.214261
2,2015-12-31,22.641818,21.484422,22.082936,21.675364,NaN,21.780604,22.463184,21.980402,24.330182,...,22.841172,23.111867,23.118971,22.917925,19.702996,19.995891,20.743809,22.527387,22.307677,24.298158
3,2016-03-31,22.621545,21.403324,21.978921,21.432638,NaN,21.806282,23.002513,21.938649,24.336524,...,22.783050,22.934486,22.928820,22.687047,19.600312,19.881795,20.552382,22.308025,22.255309,24.139839
4,2016-06-30,22.813754,21.357774,22.009083,21.282396,NaN,21.716364,23.225868,21.818938,24.365696,...,22.698619,22.757251,22.772665,22.491190,19.342428,19.769476,20.609633,22.489305,22.059406,24.219235


### 2.4 Value (Book to Price Ratio)

In [28]:
# Only include data within the specified date range
book_to_price_file_subsetted = book_to_price_file[(book_to_price_file[date_column] >= start_date) & (book_to_price_file[date_column] <= end_date) ].reset_index(drop=True)

# Set date as index and sort
book_to_price_file_subsetted = book_to_price_file_subsetted.sort_values(date_column).set_index(date_column)

# Resample to quarterly frequency, taking the last available price to book value in each quarter
book_to_price_quarterly = book_to_price_file_subsetted.resample("QE").last()

# Drop rows with all NaN values
book_to_price_quarterly = book_to_price_quarterly.dropna(how="all")

# Replace specified values with NaN
book_to_price_quarterly = replaceValuesWithNaN(book_to_price_quarterly, values_to_replace=values_to_replace)

# Reset index to have date as a column again
book_to_price_quarterly = book_to_price_quarterly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
book_to_price_quarterly[date_column] = book_to_price_quarterly[date_column].dt.date

book_to_price_quarterly.head()


Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 



,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,0.163083,0.811517,-1.089233,NaN,0.307739,1.567806,0.383862,0.347991,...,0.224417,0.219431,0.599010,0.155615,NaN,0.083209,0.102951,0.359632,0.094992,0.780756
1,2015-09-30,NaN,0.129842,0.770671,-1.171899,NaN,0.269565,2.241514,0.409166,0.383242,...,0.284154,0.232499,0.799199,0.161919,0.866315,0.047322,0.115433,0.512382,0.087390,0.869284
2,2015-12-31,0.160108,0.165532,0.701942,-1.057695,NaN,0.240910,4.232580,0.340564,0.379335,...,0.243210,0.265925,0.743645,0.129755,0.988856,0.044745,0.090181,0.494564,0.078311,0.810420
3,2016-03-31,0.536723,0.178837,0.770110,-1.254215,NaN,0.234373,1.782798,0.376292,0.351117,...,0.280584,0.295539,0.848566,0.178411,1.322129,0.078541,0.111553,0.617152,0.082521,0.899677
4,2016-06-30,0.419043,0.191152,0.748281,-1.341177,NaN,0.260303,1.461129,0.424145,0.362603,...,0.278671,0.376796,1.080340,0.218267,1.711084,0.062717,0.109801,0.510600,0.111956,0.843141


### 2.5 Mkt (CAPM Beta)

#### 2.5.1 Risk Free Rate 

In [29]:
# Only include data within the specified date range
risk_free_rate_file_subsetted = risk_free_rate_file[(risk_free_rate_file[date_column] >= start_date) & (risk_free_rate_file[date_column] <= end_date) ].reset_index(drop=True)

# Convert date column to datetime
risk_free_rate_file_subsetted[date_column] = pd.to_datetime(risk_free_rate_file_subsetted[date_column])

# Set date as index and sort
risk_free_rate_file_subsetted = risk_free_rate_file_subsetted.sort_values(date_column).set_index(date_column)

# Get risk free rate at end of the month
risk_free_rate_monthly = risk_free_rate_file_subsetted.resample("ME").last()

# Drop rows with all NaN values
risk_free_rate_monthly = risk_free_rate_monthly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
risk_free_rate_monthly[date_column] = risk_free_rate_monthly[date_column].dt.date

risk_free_rate_monthly.head()

,date,risk_free_rate
0,2015-06-30,-0.000228
1,2015-07-31,-0.000214
2,2015-08-31,-0.000204
3,2015-09-30,-0.000332
4,2015-10-31,-0.000299


#### 2.5.2 Merging Data

In [30]:
# Merging monthly stock returns, index returns, and risk-free rate into a single DataFrame
merged_monthly_data = stock_returns_monthly.merge(index_returns_monthly, on=date_column, how="inner")
merged_monthly_data = merged_monthly_data.merge(risk_free_rate_monthly, on=date_column, how="inner")

skip_columns = ["risk_free_rate", date_column]

# Calculating excess returns by subtracting the risk-free rate from stock and index returns
for stock in merged_monthly_data.columns:
    if stock not in skip_columns:
        merged_monthly_data[stock] = merged_monthly_data[stock] - merged_monthly_data["risk_free_rate"]

merged_monthly_data[date_column] = pd.to_datetime(merged_monthly_data[date_column])
merged_monthly_data = merged_monthly_data.sort_values(date_column).set_index(date_column)

In [31]:
merged_monthly_data.head()

,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,ABDN.L,...,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S,.STOXX,risk_free_rate
date,,,,,,,,,,,,,,,,,,,,,
2015-07-31,NaN,-0.037416,0.086195,-0.007165,NaN,0.158719,-0.109434,0.073037,-0.015506,0.031950,...,-0.031561,0.036435,0.269986,0.342985,-0.007660,-0.069991,0.022732,0.015449,0.039710,-0.000214
2015-08-31,NaN,0.047749,-0.058316,-0.117468,NaN,-0.041078,-0.115310,-0.030237,-0.069058,-0.113130,...,-0.120965,-0.059074,-0.060858,-0.114217,-0.117256,-0.001454,-0.045842,-0.116300,-0.084515,-0.000204
2015-09-30,NaN,0.178656,0.014958,-0.161985,NaN,-0.018372,-0.268770,-0.044778,-0.079964,-0.083422,...,-0.103092,0.014031,-0.047280,0.114939,0.016520,-0.243534,0.013974,-0.101973,-0.041069,-0.000332
2015-10-31,NaN,-0.012330,0.123722,0.012300,NaN,0.128741,0.027972,0.117646,0.084275,0.124461,...,0.158327,0.076481,-0.057496,0.034728,0.177290,0.122524,0.159719,0.093046,0.079949,-0.000299
2015-11-30,0.241106,-0.096859,0.066908,-0.008409,NaN,0.084797,-0.239638,0.057342,0.046122,0.006296,...,0.060384,0.008040,-0.001101,-0.006443,0.054484,0.114696,0.022670,0.037308,0.026875,-0.000349


#### 2.5.3 Estimating Beta

In [32]:
def calculate_rolling_beta(merged_data, stock_column, index_column, window_size=12, min_periods=12):
    """
    Calculate rolling beta for a given stock against a market index.
    """

    # Calculate rolling covariance between stock returns and index returns as well as rolling variance of index returns
    rolling_covariance = merged_data[stock_column].rolling(window=window_size, min_periods=min_periods).cov(merged_data[index_column])
    rolling_variance = merged_data[index_column].rolling(window=window_size, min_periods=min_periods).var()
    
    # Calculate rolling beta as the ratio of rolling covariance to rolling variance
    rolling_beta = rolling_covariance / rolling_variance

    return rolling_beta

In [33]:
# Creates empty DataFrame to store betas
betas_file = pd.DataFrame(index=merged_monthly_data.index, columns = merged_monthly_data.columns, dtype=float)

skip_columns = [".STOXX", "risk_free_rate"]

# Calculate rolling beta for each stock
for stock in merged_monthly_data.columns:
    if stock not in skip_columns:

        # Calculate rolling beta and store in betas DataFrame
        betas_file[stock] = calculate_rolling_beta(merged_monthly_data, stock, ".STOXX", window_size=beta_rolling_window, min_periods=beta_rolling_window)

# Resample to quarterly frequency, taking the last available price to book value in each quarter
betas_file_quarterly = betas_file.resample("QE").last()

# Drop rows with all NaN values
betas_file_quarterly = betas_file_quarterly.dropna(how="all")

betas_file_quarterly.drop(columns=["risk_free_rate", ".STOXX"], inplace=True)

# Reset index to have date as a column again
betas_file_quarterly = betas_file_quarterly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
betas_file_quarterly[date_column] = betas_file_quarterly[date_column].dt.date

In [34]:
betas_file_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2016-06-30,NaN,-0.311763,1.470557,0.974708,NaN,1.422778,0.558792,1.229154,0.988284,...,0.959105,1.541438,1.483993,0.651430,NaN,1.373208,1.530483,1.571520,1.117353,1.316710
1,2016-09-30,NaN,0.416527,1.715794,0.860749,NaN,1.385692,0.117803,1.509726,1.120462,...,0.913533,1.938175,1.711213,1.556824,NaN,0.986244,1.661987,1.847935,1.592932,1.139748
2,2016-12-31,1.233070,1.103619,1.766840,2.028220,NaN,1.331483,-0.848944,1.822476,1.010148,...,0.872873,2.549140,1.378012,2.339208,NaN,1.317678,1.370256,1.296808,1.544095,1.130264
3,2017-03-31,-0.083477,1.438449,1.748764,2.763601,NaN,1.273385,-1.870526,2.077076,0.885721,...,0.714760,2.976728,0.889741,1.981394,NaN,0.453373,0.258156,-0.940272,2.152576,-0.089459
4,2017-06-30,0.492727,1.557440,1.506942,1.619220,NaN,1.521722,-1.275468,1.482829,0.798989,...,0.993511,2.075679,0.015130,2.094929,0.126083,-0.271427,0.729154,-1.483983,1.685053,0.038090


### 2.6 Profitability (Operating Income & Total Equity)

In [35]:
def calculate_profitability_ratio(operating_income, total_equity):
    """
    Calculate profitability ratio as operating income divided by total equity.
    """

    stock_list = operating_income.columns.intersection(total_equity.columns)

    # Subset both DataFrames to only include stock list
    operating_income = operating_income[stock_list]
    total_equity = total_equity[stock_list]
    
    # profitability ratio per quarter per stock
    profitability_ratio = (operating_income / total_equity).reset_index()  

    return profitability_ratio

In [36]:
# Prepare operating income and total equity data
operating_income = prep_financial_data(operating_income_ttm_file)
total_equity = prep_financial_data(total_equity_ttm_file)

# Calculate profitability ratio
profitability_ratio = calculate_profitability_ratio(operating_income, total_equity)

profitability_ratio[date_column] = profitability_ratio[date_column].dt.date

Replaced 7 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
BOK.L^C18    5
PTNL.AS      1
DSFIR.AS     1 

Replaced 1 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
INDV.L^G25    1 



In [37]:
profitability_ratio.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,0.232679,0.137560,NaN,NaN,0.204159,NaN,NaN,0.265845,...,NaN,0.241582,0.202796,NaN,NaN,-0.725029,0.106796,NaN,NaN,0.183084
1,2015-09-30,NaN,0.248684,0.139631,NaN,NaN,0.220872,NaN,NaN,0.281342,...,NaN,0.241582,0.196754,0.092479,NaN,-1.031109,0.107157,0.097149,NaN,0.195276
2,2015-12-31,NaN,0.248023,0.135699,NaN,NaN,0.218858,NaN,NaN,0.254391,...,0.272997,0.241582,0.230404,0.072088,NaN,-1.419470,0.132165,0.097149,NaN,0.161779
3,2016-03-31,0.189099,0.195973,0.135699,NaN,NaN,0.213582,-0.304424,0.194845,0.210552,...,0.266697,0.241582,0.197303,0.070395,NaN,-0.322431,0.137653,0.097149,0.501747,0.119760
4,2016-06-30,0.199238,0.159196,0.079054,-0.15593,NaN,0.222609,-0.304424,0.194845,0.200431,...,0.283723,0.210897,0.185364,0.063131,NaN,-0.540143,0.147993,0.097149,0.501747,0.102465


### 2.7 Investment (Total Assets Reported)

In [ ]:
total_assets_reported = prep_financial_data(total_assets_reported_ttm_file)

# Calculate quarterly total asset growth rate for each stock
total_assets_growth_rate = total_assets_reported.pct_change(fill_method=None).dropna(how="all")

# Reset index to have date as a column again
total_assets_growth_rate = total_assets_growth_rate.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
total_assets_growth_rate[date_column] = total_assets_growth_rate[date_column].dt.date

Replaced 0 values ([-1, 0, 1]) with NaN.

Replacements per column (only columns with >=1 replacement):
Series([], ) 



In [39]:
total_assets_growth_rate.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-09-30,NaN,0.608372,-0.044410,NaN,NaN,-0.029165,NaN,NaN,-0.010247,...,0.066591,0.000000,0.013712,0.002477,NaN,-0.092078,0.007051,NaN,NaN,-0.001581
1,2015-12-31,NaN,-0.190182,0.020893,NaN,NaN,0.021798,NaN,NaN,-0.021666,...,-0.002312,0.000000,-0.015105,0.087456,0.0,0.036608,-0.002398,0.000000,NaN,-0.034786
2,2016-03-31,-0.027159,0.008142,0.000000,NaN,NaN,0.077961,NaN,NaN,-0.009864,...,-0.003566,0.000000,-0.042265,0.001846,0.0,0.184692,0.156929,0.000000,NaN,-0.013110
3,2016-06-30,0.037793,-0.090261,-0.033443,NaN,NaN,0.002447,0.0,0.0,0.011075,...,0.012167,0.179844,0.057672,0.037751,0.0,-0.157147,0.035262,0.000000,0.0,0.048847
4,2016-09-30,-0.075030,-0.023687,-0.007953,0.0,NaN,0.022613,0.0,0.0,0.004520,...,-0.037652,0.000000,0.024692,0.002550,0.0,-0.107476,0.002734,0.026202,0.0,-0.014203


### 2.8 Price Momentum

In [40]:
momentum_monthly = price_close_stocks_monthly.copy()

# Calculate 6-month momentum
momentum_monthly = momentum_monthly.shift(1) / momentum_monthly.shift(7) - 1

# Resample to quarterly frequency, taking values from March, June, September, and December
momentum_quarterly = momentum_monthly[momentum_monthly.index.month.isin([3, 6, 9, 12])]

# Reset index to have date as a column again
momentum_quarterly = momentum_quarterly.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
momentum_quarterly[date_column] = momentum_quarterly[date_column].dt.date


In [41]:
momentum_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2015-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2016-03-31,NaN,-0.072049,-0.033821,-0.204735,NaN,0.063581,-0.397439,0.033742,-0.042066,...,0.005688,-0.240061,-0.098954,-0.013356,-0.33467,-0.086690,0.009442,-0.431839,0.033578,-0.200912
4,2016-06-30,0.097164,-0.130933,-0.036842,-0.026381,NaN,-0.050580,0.340393,0.014720,0.041707,...,-0.124354,-0.154667,-0.261246,-0.179283,NaN,-0.173472,-0.065621,-0.181038,-0.186026,-0.127514


### 2.9 ESG

In [42]:
def getESG(data, quarterly, forward_filling):

    # Create a copy of the original data to avoid modifying it directly
    esg = data.copy()

    # Set date as index and sort
    esg = esg.sort_values(date_column).set_index(date_column)

    # Set frequency string based on whether quarterly or monthly frequency is desired
    freq = "QE" if quarterly else "ME"

    # Resample to quarterly or monthly frequency, taking the last available ESG score in each quarter / month
    esg = esg.resample(freq).last()

    # Create a complete date range from the earliest date in the ESG data to the specified end date with the appropriate frequency (quarterly or monthly)
    full_index = pd.date_range(
        start=pd.to_datetime(start_date_esg),
        end=pd.to_datetime(end_date),
        freq=freq
    )

    # Reindex the ESG DataFrame to the complete date range, which will introduce NaN values for any missing dates in the original data
    esg = esg.reindex(full_index)

    # If forward_filling is True, fill forward missing values (The most recent available data is used to fill missing values)
    if forward_filling:
        esg = fill_forward_data(esg)
    
    # Reset index to have date as a column again
    esg = esg.reset_index().rename(columns={"index": date_column})

    # Only include data within the specified date range
    esg = esg[(esg[date_column] >= start_date_esg) & (esg[date_column] <= end_date)].reset_index(drop=True)
    
    # Convert date column to date only (remove time component)
    esg[date_column] = esg[date_column].dt.date

    return esg

In [43]:
# For regressions
esg_ff_monthly = getESG(esg_score_file, quarterly=False, forward_filling=True)
esg_ff_quarterly = getESG(esg_score_file, quarterly=True, forward_filling=True)

# For calculating ESG momentum
esg_nff_monthly = getESG(esg_score_file, quarterly=False, forward_filling=False)
esg_nff_quarterly = getESG(esg_score_file, quarterly=True, forward_filling=False)

In [44]:
esg_ff_monthly.tail()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
125,2025-05-31,66.638959,70.815077,69.376452,40.042428,61.883586,70.139053,73.670825,63.63931,89.767579,...,81.544057,77.541464,79.712853,73.739853,14.943958,61.109133,48.872749,44.579421,50.97968,82.930424
126,2025-06-30,66.638959,70.815077,69.376452,40.042428,61.883586,70.139053,73.670825,63.63931,89.767579,...,81.544057,77.541464,79.712853,73.739853,14.943958,61.109133,48.872749,44.579421,50.97968,82.930424
127,2025-07-31,66.638959,70.815077,69.376452,40.042428,61.883586,70.139053,73.670825,63.63931,89.767579,...,81.544057,77.541464,79.712853,73.739853,14.943958,61.109133,48.872749,44.579421,50.97968,82.930424
128,2025-08-31,66.638959,70.815077,69.376452,40.042428,61.883586,70.139053,73.670825,63.63931,89.767579,...,81.544057,77.541464,79.712853,73.739853,14.943958,61.109133,48.872749,44.579421,50.97968,82.930424
129,2025-09-30,66.638959,70.815077,69.376452,40.042428,61.883586,70.139053,73.670825,63.63931,89.767579,...,81.544057,77.541464,79.712853,73.739853,14.943958,61.109133,48.872749,44.579421,50.97968,82.930424


In [45]:
esg_ff_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2014-12-31,NaN,NaN,61.082074,NaN,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,NaN,62.982869,NaN,NaN,NaN,NaN,NaN,33.676215,67.859389
1,2015-03-31,NaN,NaN,61.082074,6.898408,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,77.323048,62.982869,NaN,NaN,NaN,NaN,NaN,33.676215,67.859389
2,2015-06-30,NaN,NaN,61.082074,6.898408,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,77.323048,62.982869,NaN,NaN,NaN,NaN,NaN,33.676215,67.859389
3,2015-09-30,NaN,NaN,61.082074,6.898408,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,77.323048,62.982869,NaN,NaN,NaN,NaN,46.637297,33.676215,67.859389
4,2015-12-31,52.449942,NaN,59.459100,6.898408,NaN,NaN,81.434442,17.964003,88.899243,...,75.294219,77.323048,66.670726,42.209841,NaN,NaN,NaN,46.637297,23.662956,66.916603


In [46]:
esg_nff_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2014-12-31,NaN,NaN,61.082074,NaN,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,NaN,62.982869,NaN,NaN,NaN,NaN,NaN,NaN,67.859389
1,2015-01-31,NaN,NaN,NaN,6.898408,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,77.323048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [47]:
esg_nff_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2014-12-31,NaN,NaN,61.082074,NaN,NaN,NaN,88.886685,12.356561,92.893651,...,66.782415,NaN,62.982869,NaN,NaN,NaN,NaN,NaN,33.676215,67.859389
1,2015-03-31,NaN,NaN,NaN,6.898408,NaN,NaN,NaN,NaN,NaN,...,NaN,77.323048,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,46.637297,NaN,NaN
4,2015-12-31,52.449942,NaN,59.459100,NaN,NaN,NaN,81.434442,17.964003,88.899243,...,75.294219,NaN,66.670726,42.209841,NaN,NaN,NaN,NaN,23.662956,66.916603


### 2.10 ESG Momentum

In [48]:
def getESGMomentum(prepared_esg_data, forward_filling):

    esg_momentum = prepared_esg_data.copy()

    # Set date as index and sort
    esg_momentum = esg_momentum.sort_values(date_column).set_index(date_column)

    # Calculate ESG momentum as the change in ESG score from the last available ESG score
    prev_valid = esg_momentum.ffill().shift(1)

    esg_momentum = (esg_momentum - prev_valid).where(esg_momentum.notna())

    # If forward_filling is True, fill forward missing values (The most recent available data is used to fill missing values)
    if forward_filling:
        esg_momentum = fill_forward_data(esg_momentum)
    
    # Reset index to have date as a column again
    esg_momentum = esg_momentum.reset_index().rename(columns={"index": date_column})

    # Convert date column to datetime
    esg_momentum[date_column] = pd.to_datetime(esg_momentum[date_column]).dt.date

    return esg_momentum

In [49]:
esg_momentum_quarterly = getESGMomentum(esg_nff_quarterly, forward_filling=True) # Used for panel regressions
esg_momentum_monthly = getESGMomentum(esg_nff_monthly, forward_filling=True) # Used to form portfolios in business cycle analysis

In [50]:
esg_momentum_quarterly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2014-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-06-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-12-31,NaN,NaN,-1.622974,NaN,NaN,NaN,-7.452243,5.607441,-3.994408,...,8.511805,NaN,3.687856,NaN,NaN,NaN,NaN,NaN,-10.013258,-0.942786


In [51]:
esg_momentum_monthly.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2014-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-02-28,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.11 Constituents & Presence Matrix

The data retrieved from Refinitiv contains duplicate constituent entries. So there are cases where one company appears with two different RICs. In those cases, I need to identify if any of those duplicates are indicated to be present in the STOXX 600 index at the same time. In that case, I would have to eliminate one of them as in reality they don't appear twice. In the other cases where one is always 0 while the other one is 1, there is no problem. 

In [52]:
# Only include data within the specified date range
presence_matrix = presence_matrix_file[presence_matrix_file[date_column] >= constituent_cut_off_date].reset_index(drop=True)

# Identify companies with duplicate entries
dup_companies = (
    constituents_file["Company_Name"]
    .value_counts()
    .loc[lambda s: s > 1]
    .index
)

# Get mapping of company names to their corresponding RICs
company_to_rics = (
    constituents_file.loc[constituents_file["Company_Name"].isin(dup_companies)]
    .groupby("Company_Name")["RIC"]
    .apply(list)
    .to_dict()
)

results = []
overlap_rows = {}

# Loop over all companies with duplicate entries
for company, rics in company_to_rics.items():

    # Get RICs that are present in the presence matrix
    rics_present = [r for r in rics if r in presence_matrix.columns]

    # If less than 2 RICs are present, there cannot be any overlap
    if len(rics_present) < 2:
        results.append({
            "Company_Name": company,
            "n_rics_total": len(rics),
            "n_rics_in_presence_matrix": len(rics_present),
            "any_overlap": False,
            "n_overlap_rows": 0,
            "rics": rics_present
        })

        continue

    # Get counts of how many rows are equal to 1 for the RICs present
    counts = presence_matrix[rics_present].eq(1).sum(axis=1)

    # Create a mask for rows where at least two RICs have a value of 1
    mask = counts.ge(2)

    # Determine if there is any overlap
    any_ov = mask.any()

    # Store results
    results.append({
        "Company_Name": company,
        "n_rics_total": len(rics),
        "n_rics_in_presence_matrix": len(rics_present),
        "any_overlap": any_ov,
        "n_overlap_rows": int(mask.sum()),
        "rics": rics_present
    })

    if any_ov:
        overlap_rows[company] = presence_matrix.index[mask].tolist()

report = (
    pd.DataFrame(results)
    .sort_values(by=["any_overlap", "n_overlap_rows"], ascending=[False, False])
    .reset_index(drop=True)
)

report

,Company_Name,n_rics_total,n_rics_in_presence_matrix,any_overlap,n_overlap_rows,rics
0,Rolls-Royce Holdings PLC,2,2,True,1,"[RR.L, RRn.L^K20]"
1,UniCredit SpA,2,2,True,1,"[CRDI.MI, CRDI_r.MI^B17]"
2,ams-OSRAM AG,2,2,True,1,"[AMS.S, AMS_r.S^C20]"
3,Adevinta AS,2,2,False,0,"[ADEA.OL^F24, ADEA.OL^J19]"
4,CRH PLC,2,2,False,0,"[CRH.I^I23, CRH.L]"
5,Chocoladefabriken Lindt & Spruengli AG,2,2,False,0,"[LISN.S, LISP.S]"
6,Deutsche Boerse AG,2,2,False,0,"[DB11.DE^D17, DB1Gn.DE]"
7,Exor NV,2,2,False,0,"[EXOR.AS, EXOR.MI^I22]"
8,Ferguson Enterprises Inc,2,2,False,0,"[FERG.L, FERG.N]"
9,Flutter Entertainment PLC,2,2,False,0,"[FLTRF.I^A24, FLTRF.L]"


There are three companies for which there are dates where two RICs are indicated to be present in the index at the same time. Each RIC combination has only one date where this is the case. The companies are:

- Rolls-Royce Holdings PLC
- UniCredit SpA
- ams-OSRAM AG

All other RICs that are the same company don't have an overlap on any date. For that reason, I will not make any adjustments for those as they will never be considered for further analysis on the same date. 

Now I am checking on how many dates each RIC that is part of the duplicate list is indicated to be present in the index in general.

In [53]:
# Get list of all RICs for companies with duplicate entries
dup_rics = (
    constituents_file.loc[constituents_file["Company_Name"].isin(dup_companies), "RIC"]
    .dropna()
    .unique()
    .tolist()
)

# Filter RICs to only those present in the presence matrix
dup_rics_present = [r for r in dup_rics if r in presence_matrix.columns]

# Count number of ones in presence matrix for each RIC
ric_one_counts = presence_matrix.loc[:, dup_rics_present].eq(1).sum(axis=0)

# Create long format DataFrame with RICs and their corresponding counts of ones
ric_ones_long = (
    constituents_file.loc[
        constituents_file["RIC"].isin(dup_rics_present),
        ["Company_Name", "RIC"]
    ]
    .drop_duplicates()
    .assign(ones_count=lambda d: d["RIC"].map(ric_one_counts).fillna(0).astype(int)) # Map counts to RICs
    .sort_values(["Company_Name", "ones_count", "RIC"], ascending=[True, False, True]) # Sort by Company_Name, then by ones_count descending, then by RIC
    .reset_index(drop=True)
)

ric_ones_long

,Company_Name,RIC,ones_count
0,Adevinta AS,ADEA.OL^F24,1043
1,Adevinta AS,ADEA.OL^J19,1
2,CRH PLC,CRH.I^I23,2665
3,CRH PLC,CRH.L,280
4,Chocoladefabriken Lindt & Spruengli AG,LISN.S,2301
5,Chocoladefabriken Lindt & Spruengli AG,LISP.S,1259
6,Deutsche Boerse AG,DB1Gn.DE,3328
7,Deutsche Boerse AG,DB11.DE^D17,232
8,Exor NV,EXOR.MI^I22,2301
9,Exor NV,EXOR.AS,1259


For all three companies with RICs that have instances where they appear on the same date, the second RIC only appears once at all. For that reason, I will drop those RICs that only appear once.

In [54]:
# RICs of the three companies with duplicate date entries
rics_column_list = [
    "RR.L",
    "RRn.L^K20",
    "CRDI.MI",
    "CRDI_r.MI^B17",
    "AMS.S",
    "AMS_r.S^C20"
]

# Filter ric_ones_long to only include specified RICs
ric_ones_long_three_companies = ric_ones_long[ric_ones_long["RIC"].isin(rics_column_list)]

ric_ones_long_three_companies

,Company_Name,RIC,ones_count
22,Rolls-Royce Holdings PLC,RR.L,3560
23,Rolls-Royce Holdings PLC,RRn.L^K20,1
26,UniCredit SpA,CRDI.MI,3560
27,UniCredit SpA,CRDI_r.MI^B17,1
32,ams-OSRAM AG,AMS.S,2756
33,ams-OSRAM AG,AMS_r.S^C20,1


In [55]:
# Drop specified columns from presence_matrix
drop_columns = [
    "RRn.L^K20",
    "CRDI_r.MI^B17",
    "AMS_r.S^C20"
]

# Drop specified columns from presence_matrix
presence_matrix = presence_matrix.drop(columns=drop_columns)

Now I will check if all the RICs in the presence_matrix appear in every other dataframe and if any columns in other dataframes don't appear in the presence_matrix

-> This check will show any other potential naming mismatches between the dataframes

In [56]:
def stock_presence_report(
    presence_matrix_df: pd.DataFrame,
    df_dict: dict[str, pd.DataFrame],
):

    # Lookup mapping from RIC to Company Name
    ric_to_company = (
        constituents_file[["RIC", "Company_Name"]]
        .dropna(subset=["RIC"])
        .drop_duplicates(subset=["RIC"])
        .set_index("RIC")["Company_Name"]
    )

    # Get list of stocks in presence matrix
    present_matrix_stocks = [stock for stock in presence_matrix_df.columns if stock != date_column]
    present_matrix_set = set(present_matrix_stocks)

    df_names = list(df_dict.keys())

    presence_matrix_cross_check_rows = []

    # Loop over all stocks in presence matrix
    for stock in present_matrix_stocks:
        present = [name for name, df in df_dict.items() if stock in df.columns] # List of DataFrames where stock is present
        absent = [name for name in df_names if name not in present] # List of DataFrames where stock is absent

        if absent:
            # Append results to rows list
            presence_matrix_cross_check_rows.append({
                "company_name": ric_to_company.get(stock, pd.NA),
                "stock": stock,
                "present_in": present,
                "absent_in": absent
            })
        
    # Create DataFrame from rows list
    report_presence_matrix_missing_in_df = pd.DataFrame(presence_matrix_cross_check_rows).sort_values(by="company_name").reset_index(drop=True)

    missing_in_presence_matrix_row = []

    # Loop over all DataFrames to check for stocks missing in presence matrix
    for name, df in df_dict.items():
        # Get list of stocks in current DataFrame
        df_stocks = [stock for stock in df.columns if stock != date_column]

        # Loop over all stocks in current DataFrame
        for stock in df_stocks:

            # Skip dropped columns as they naturally won't appear in presence matrix
            if stock not in drop_columns:

                # Check if stock is missing in presence matrix
                if stock not in present_matrix_set:
                    missing_in_presence_matrix_row.append({
                        "dataframe": name,
                        "stock": stock
                    })

    print(missing_in_presence_matrix_row)
    
    if missing_in_presence_matrix_row:
        report_missing_in_presence_matrix_df = pd.DataFrame(missing_in_presence_matrix_row).sort_values(by=["dataframe", "stock"]).reset_index(drop=True)
    else: 
        report_missing_in_presence_matrix_df = pd.DataFrame(columns=["dataframe", "stock"])

    return report_presence_matrix_missing_in_df, report_missing_in_presence_matrix_df

In [57]:
comparison_result = {
    "price_close_stocks_file": price_close_stocks_file,
    "free_float_mcap_file": free_float_mcap_file,
    "outstanding_mcap_file": outstanding_mcap_file,
    "outstanding_shares_file": outstanding_shares_file,
    "book_to_price_file": book_to_price_file,
    "operating_income_ttm_file": operating_income_ttm_file,
    "total_equity_ttm_file": total_equity_ttm_file,
    "total_assets_reported_ttm_file": total_assets_reported_ttm_file,
    "esg_score_file": esg_score_file
}

report_presence_matrix_missing_in_df, report_missing_in_presence_matrix_df = stock_presence_report(
    presence_matrix_df=presence_matrix,
    df_dict= comparison_result
)

[{'dataframe': 'esg_score_file', 'stock': 'AHT.L'}, {'dataframe': 'esg_score_file', 'stock': 'DWL.L'}, {'dataframe': 'esg_score_file', 'stock': 'PHNX.L'}]


In [58]:
report_presence_matrix_missing_in_df

,company_name,stock,present_in,absent_in
0,ABN Amro Bank NV,ABNd.AS,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
1,AIB Group PLC,AIBG.I,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
2,AL Sydbank A/S,ALSYDB.CO,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
3,Aareal Bank AG,ARLn.DE^K23,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
4,Abivax SA,ABVX.PA,"[price_close_stocks_file, free_float_mcap_file...",[esg_score_file]
...,...,...,...,...
78,Unicaja Banco SA,UNI.MC,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
79,Unione di Banche Italiane SpA,UBI.MI^J20,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]
80,Verisure PLC,VSURE.ST,"[price_close_stocks_file, free_float_mcap_file...",[esg_score_file]
81,Virgin Money UK PLC,VMUK.L^J24,"[price_close_stocks_file, free_float_mcap_file...",[operating_income_ttm_file]


In [59]:
report_missing_in_presence_matrix_df

,dataframe,stock
0,esg_score_file,AHT.L
1,esg_score_file,DWL.L
2,esg_score_file,PHNX.L


The comparison showed that there are 83 RICs that don't appear in one or more of the following dataframes:

- operating_income_ttm_file
- esg_score_file
- book_to_price_file
- free_float_mcap_file

However, most of them don't appear in operating_income_ttm_file. 

In [60]:
# Set date column as date
presence_matrix[date_column] = pd.to_datetime(presence_matrix[date_column]).dt.date

presence_matrix.head()

,date,1COVG.DE^L25,1U1.DE,A2.MI,AAAA.L^C21,AAF.L,AAK.ST,AAL.L,AALB.AS,ABBN.S,...,WRT1V.HE,WTB.L,YAR.OL,ZALG.DE,ZEG.L,ZELA.CO,ZO1G.DE^A22,ZODC.PA^C18,ZOT.MC^E22,ZURN.S
0,2016-06-01,1,0,0,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
1,2016-06-02,1,0,0,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
2,2016-06-03,1,0,0,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
3,2016-06-04,1,0,0,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1
4,2016-06-05,1,0,0,1,0,0,1,1,1,...,1,1,1,1,0,0,0,1,1,1


### 2.12 Dividend Yield (DIV)

In [61]:
def rename_columns(file, suffix_value):
    """
    Renames columns in the DataFrame by appending a suffix, except for the date column.
    """
    
    for column in file.columns:
        if column != date_column:
            file = file.rename(columns={column: f"{column}_{suffix_value}"})

    return file

In [62]:
stocks = price_close_stocks_daily.columns.tolist()

stock_prices_daily = price_close_stocks_daily.copy()
index_prices_daily = price_close_index_daily.copy()
free_float_mcap_daily = free_float_mcap_file.copy()
dividends = adjusted_gross_dividend_amount_file.copy()

free_float_mcap_daily[date_column] = pd.to_datetime(free_float_mcap_daily[date_column]).dt.date
dividends[date_column] = pd.to_datetime(dividends[date_column]).dt.date

# Rename columns to avoid conflicts during merging
stock_prices_daily = rename_columns(stock_prices_daily, "prices")
index_prices_daily = rename_columns(index_prices_daily, "prices")
free_float_mcap_daily = rename_columns(free_float_mcap_daily, "mcap")
dividends = rename_columns(dividends, "dividends")
presence = rename_columns(presence_matrix, "presence")

# Merge dataframes on date column
merged_dividend_yield_data = stock_prices_daily.merge(index_prices_daily, on=date_column, how="outer")
merged_dividend_yield_data = merged_dividend_yield_data.merge(free_float_mcap_daily, on=date_column, how="outer")
merged_dividend_yield_data = merged_dividend_yield_data.merge(dividends, on=date_column, how="outer")
merged_dividend_yield_data = merged_dividend_yield_data.merge(presence, on=date_column, how="outer")

# Only include data within the specified date range
merged_dividend_yield_data = merged_dividend_yield_data[merged_dividend_yield_data[date_column] >= pd.to_datetime(start_date).date()].reset_index(drop=True)

print("Check if the merged dataframe has the correct number of columns:")
print(f"Expected number of columns: {len(stock_prices_daily.columns) + len(index_prices_daily.columns) + len(free_float_mcap_daily.columns) + len(dividends.columns) + len(presence.columns) - 4}")  # Subtract 4 to account for duplicate date columns
print(f"Actual number of columns: {len(merged_dividend_yield_data.columns)}\n")

Check if the merged dataframe has the correct number of columns:
Expected number of columns: 3796
Actual number of columns: 3796



The dataframe "merged_dividend_yield_data" starts on 2015-06-01 to allow a one year window for calculation of dividend yield. For that reason, I set every value of the "{stock}_presence" column before 2016-06-01 to be equal to the value of that column on 2016-06-01. 

In [63]:
cutoff = pd.Timestamp(constituent_cut_off_date).date()

# Get list of presence columns
presence_cols = [c for c in merged_dividend_yield_data.columns if c.endswith("_presence")]

# Get the values at the cut-off date for all presence columns
cutoff_vals = (
    merged_dividend_yield_data.loc[merged_dividend_yield_data[date_column] == cutoff, presence_cols]
      .iloc[0]
)

# Create a mask for dates before the cut-off date
mask = merged_dividend_yield_data[date_column] < cutoff
merged_dividend_yield_data.loc[mask, presence_cols] = cutoff_vals.values

In [64]:
merged_dividend_yield_data.head()

,date,1COVG.DE^L25_prices,1U1.DE_prices,A2.MI_prices,AAAA.L^C21_prices,AAF.L_prices,AAK.ST_prices,AAL.L_prices,AALB.AS_prices,ABBN.S_prices,...,WRT1V.HE_presence,WTB.L_presence,YAR.OL_presence,ZALG.DE_presence,ZEG.L_presence,ZELA.CO_presence,ZO1G.DE^A22_presence,ZODC.PA^C18_presence,ZOT.MC^E22_presence,ZURN.S_presence
0,2015-06-01,NaN,41.610,1.160,5.741943,NaN,9.881239,13.251426,28.430,19.205558,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
1,2015-06-02,NaN,42.635,1.169,5.673418,NaN,9.649951,13.697202,28.215,19.133646,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
2,2015-06-03,NaN,42.600,1.156,5.703756,NaN,9.688753,13.537380,28.080,19.276204,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
3,2015-06-04,NaN,42.240,1.134,5.715221,NaN,9.575414,13.048484,27.640,20.033999,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0
4,2015-06-05,NaN,41.710,1.116,5.667666,NaN,9.415398,13.252652,27.295,19.925396,...,1.0,1.0,1.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0


In [65]:
missing_columns = []

# Check for duplicated columns after merging
dupes = merged_dividend_yield_data.columns[merged_dividend_yield_data.columns.duplicated()]
dupes

Index([], dtype='object')

In [66]:
# Iterate over each stock to check for presence and set market cap to NaN where presence is not 1
# This automatically ensures that dividend yield calculations only consider stocks that are present in the index
for stock in stocks:

    # Skip date column and dropped columns
    if stock != date_column and stock not in drop_columns:
        mcap_col = f"{stock}_mcap"
        pres_col = f"{stock}_presence"

        # Check if columns exist in the merged DataFrame
        if mcap_col not in merged_dividend_yield_data.columns:
            print(f"[SKIP] {stock}: missing column -> {mcap_col}")
            missing_columns.append(mcap_col)
            continue

        if pres_col not in merged_dividend_yield_data.columns:
            print(f"[SKIP] {stock}: missing column -> {pres_col}")
            missing_columns.append(pres_col)
            continue

        # Set market cap to NaN where presence is not 1
        merged_dividend_yield_data.loc[
            merged_dividend_yield_data[pres_col] != 1, mcap_col
        ] = np.nan

In [ ]:
index_price_col = ".STOXX_prices"
total_mcap_col = "total_daily_mcap"

weights = {}
div_points = {}

# Calculate total daily market capitalization across all stocks for each date with those stocks that are present on that date
merged_dividend_yield_data[total_mcap_col] = merged_dividend_yield_data[[col for col in merged_dividend_yield_data.columns if col.endswith('_mcap')]].sum(axis=1)

for stock in stocks:
    if stock != date_column and stock not in drop_columns:
        # Define column names
        stock_price_col = f"{stock}_prices"
        stock_mcap_col = f"{stock}_mcap"
        dividend_col = f"{stock}_dividends"

        required_cols = [
            stock_price_col,
            stock_mcap_col,
            dividend_col,
        ]

        # Check existence
        missing = [c for c in required_cols if c not in merged_dividend_yield_data.columns]

        # Skip stock if any required column is missing
        # Missing dividend columns are expected for stocks that don't pay dividends
        if missing:
            print(f"[SKIP] {stock}: missing column(s) -> {missing}")
            missing_columns.extend(missing)
            continue

        stock_index_weight = f"{stock}_index_weight"
        dividend_points_col = f"{stock}_dividend_points"

        # Calculate index weight
        w = merged_dividend_yield_data[stock_mcap_col] / merged_dividend_yield_data[total_mcap_col]
        weights[stock_index_weight] = w

        # Calculate dividend points
        div_points[dividend_points_col] = merged_dividend_yield_data[dividend_col] * (w * merged_dividend_yield_data[index_price_col] / merged_dividend_yield_data[stock_price_col])

# Append weights and dividend points to the merged DataFrame
merged_dividend_yield_data = pd.concat([merged_dividend_yield_data, pd.DataFrame(weights), pd.DataFrame(div_points)], axis=1)

merged_dividend_yield_data.head()

[SKIP] ABLX.BR^F18: missing column(s) -> ['ABLX.BR^F18_dividends']
[SKIP] ABVX.PA: missing column(s) -> ['ABVX.PA_dividends']
[SKIP] ADEA.OL^F24: missing column(s) -> ['ADEA.OL^F24_dividends']
[SKIP] ADEA.OL^J19: missing column(s) -> ['ADEA.OL^J19_dividends']
[SKIP] ADYEN.AS: missing column(s) -> ['ADYEN.AS_dividends']
[SKIP] AG1G.DE: missing column(s) -> ['AG1G.DE_dividends']
[SKIP] AIRF.PA: missing column(s) -> ['AIRF.PA_dividends']
[SKIP] ALEP.WA: missing column(s) -> ['ALEP.WA_dividends']
[SKIP] ALFEN.AS: missing column(s) -> ['ALFEN.AS_dividends']
[SKIP] AMRZ.S: missing column(s) -> ['AMRZ.S_dividends']
[SKIP] ARGX.BR: missing column(s) -> ['ARGX.BR_dividends']
[SKIP] ASMDEEb.ST: missing column(s) -> ['ASMDEEb.ST_dividends']
[SKIP] ATCA.AS^A21: missing column(s) -> ['ATCA.AS^A21_dividends']
[SKIP] AUTO.OL: missing column(s) -> ['AUTO.OL_dividends']
[SKIP] BALDb.ST: missing column(s) -> ['BALDb.ST_dividends']
[SKIP] BAVA.CO: missing column(s) -> ['BAVA.CO_dividends']
[SKIP] BTG.L^H

,date,1COVG.DE^L25_prices,1U1.DE_prices,A2.MI_prices,AAAA.L^C21_prices,AAF.L_prices,AAK.ST_prices,AAL.L_prices,AALB.AS_prices,ABBN.S_prices,...,WMH.L^D21_dividend_points,WPG.L^A18_dividend_points,WPP.L_dividend_points,WRT1V.HE_dividend_points,WTB.L_dividend_points,YAR.OL_dividend_points,ZEG.L_dividend_points,ZODC.PA^C18_dividend_points,ZOT.MC^E22_dividend_points,ZURN.S_dividend_points
0,2015-06-01,NaN,41.610,1.160,5.741943,NaN,9.881239,13.251426,28.430,19.205558,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2015-06-02,NaN,42.635,1.169,5.673418,NaN,9.649951,13.697202,28.215,19.133646,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2015-06-03,NaN,42.600,1.156,5.703756,NaN,9.688753,13.537380,28.080,19.276204,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2015-06-04,NaN,42.240,1.134,5.715221,NaN,9.575414,13.048484,27.640,20.033999,...,NaN,NaN,0.023678,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2015-06-05,NaN,41.710,1.116,5.667666,NaN,9.415398,13.252652,27.295,19.925396,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [68]:
# Subset dividend yield data to only include date, dividend points, and index prices
dp_cols = [c for c in merged_dividend_yield_data.columns if c.endswith("_dividend_points")]

# Keep only relevant columns
dividend_points = merged_dividend_yield_data[[date_column] + dp_cols + [".STOXX_prices"]].copy()

In [69]:
# Sum dividend points across all stocks to get total daily dividend points
dividend_points["total_dividend_points"] = (dividend_points[dp_cols].sum(axis=1, skipna=True))

# Keep only date, total dividend points, and index prices
dividend_yield = dividend_points[[date_column, "total_dividend_points", ".STOXX_prices"]]

# Convert date column to datetime
dividend_yield[date_column] = pd.to_datetime(dividend_yield[date_column]) 

# Calculate monthly sum of total dividend points and last available index price in each month
monthly_dividend_yield = (
    dividend_yield.sort_values(date_column)
    .groupby(pd.Grouper(key = date_column, freq="ME"), as_index=True)
    .agg(
        total_dividend_points = ("total_dividend_points", "sum"),
        **{".STOXX_prices": (".STOXX_prices", "last")}
    )
)

# Calculate TTM total dividend points
monthly_dividend_yield["total_dividend_points_ttm"] = monthly_dividend_yield["total_dividend_points"].rolling(window=12).sum()

# Calculate TTM dividend yield
monthly_dividend_yield["dividend_yield_ttm"] = monthly_dividend_yield["total_dividend_points_ttm"] / monthly_dividend_yield[".STOXX_prices"]

# Reset index to have date as a column again
monthly_dividend_yield = monthly_dividend_yield.reset_index().rename(columns={"index": date_column})

# Convert date column to date only (remove time component)
monthly_dividend_yield[date_column] = monthly_dividend_yield[date_column].dt.date

# Only include data within the specified date range
monthly_dividend_yield = monthly_dividend_yield[(monthly_dividend_yield[date_column] >= pd.to_datetime(constituent_cut_off_date).date()) & (monthly_dividend_yield[date_column] <= pd.to_datetime(end_date).date())].reset_index(drop=True)

monthly_dividend_yield.head()


/tmp/ipykernel_6068/1704595468.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dividend_yield[date_column] = pd.to_datetime(dividend_yield[date_column])


,date,total_dividend_points,.STOXX_prices,total_dividend_points_ttm,dividend_yield_ttm
0,2016-06-30,1.051688,329.88,15.129034,0.045862
1,2016-07-31,0.376422,341.89,15.058976,0.044046
2,2016-08-31,1.140731,343.53,14.958598,0.043544
3,2016-09-30,0.338833,342.92,14.773672,0.043082
4,2016-10-31,0.542962,338.97,14.816858,0.043711


### 2.13 Yield (YLD)

In [70]:
three_month_yield = gov_bond_yields.copy()

# Only include data within the specified date range
three_month_yield = three_month_yield[(three_month_yield[date_column] >= start_date) & (three_month_yield[date_column] <= end_date)].reset_index(drop=True)

# Resample to keep last available value in each month
three_month_yield = three_month_yield.sort_values(date_column).set_index(date_column).resample("ME").last().reset_index()

# Set date as index and sort
three_month_yield[date_column] = pd.to_datetime(three_month_yield[date_column]).dt.date

# Keep only relevant columns
three_month_yield = three_month_yield[[date_column, "spot_rate_3m_government_bond"]]

three_month_yield.head()

,date,spot_rate_3m_government_bond
0,2015-06-30,-0.002659
1,2015-07-31,-0.002689
2,2015-08-31,-0.002540
3,2015-09-30,-0.003581
4,2015-10-31,-0.003472


### 2.14 Term Spread (TERM)

In [71]:
term_spread = gov_bond_yields.copy()

# Only include data within the specified date range
term_spread = term_spread[(term_spread[date_column] >= start_date) & (term_spread[date_column] <= end_date)].reset_index(drop=True)

# Calculate term spread as the difference between the 10-year government bond spot rate and the 3-month government bond spot rate
term_spread["term_spread"] = term_spread["spot_rate_10y_government_bond"] - term_spread["spot_rate_3m_government_bond"]

# Resample to keep last available value in each month
term_spread = term_spread.sort_values(date_column).set_index(date_column).resample("ME").last().reset_index()

# Remove timestamp of date column
term_spread[date_column] = pd.to_datetime(term_spread[date_column]).dt.date

# Keep only relevant columns
term_spread = term_spread[[date_column, "term_spread"]]

term_spread.head()

,date,term_spread
0,2015-06-30,0.012188
1,2015-07-31,0.009976
2,2015-08-31,0.010740
3,2015-09-30,0.010615
4,2015-10-31,0.009772


### 2.15 Default Spread (DEF)

In [72]:
bond_yields = bond_yields_file.copy()

# Only include data within the specified date range
bond_yields = bond_yields[(bond_yields[date_column] >= start_date) & (bond_yields[date_column] <= end_date) ].reset_index(drop=True)

# Calculate default spread
bond_yields["default_spread"] = bond_yields["BAA"] - bond_yields["AAA"]

# Resample to keep last available value in each month
bond_yields = bond_yields.sort_values(date_column).set_index(date_column).resample("ME").last().reset_index()

# Remove timestamp of date column
bond_yields[date_column] = pd.to_datetime(bond_yields[date_column]).dt.date

bond_yields.head()


,date,AAA,BAA,default_spread
0,2015-06-30,0.0419,0.0513,0.0094
1,2015-07-31,0.0415,0.0520,0.0105
2,2015-08-31,0.0404,0.0519,0.0115
3,2015-09-30,0.0407,0.0534,0.0127
4,2015-10-31,0.0395,0.0534,0.0139


### 2.16 Sector Dummy

In [73]:
industry_file.head()

,stock,industry
0,1COVG.DE^L25,Basic Materials
1,1U1.DE,Telecommunications
2,A2.MI,Utilities
3,AAAA.L^C21,Consumer Discretionary
4,AAF.L,Telecommunications


### 2.17 Country Dummy

In [74]:
country_of_headquarters_file.head()

,stock,country_of_headquarters
0,1COVG.DE^L25,Germany
1,1U1.DE,Germany
2,A2.MI,Italy
3,AAAA.L^C21,United Kingdom
4,AAF.L,United Kingdom


### 2.18 Currency Dummy

In [75]:
currency_file.head()

,stock,currency
0,1COVG.DE^L25,EUR
1,1U1.DE,EUR
2,A2.MI,EUR
3,AAAA.L^C21,GBp
4,AAF.L,GBp


### 2.19 Fama-French Data

In [76]:
fama_french_data[date_column] = pd.to_datetime(fama_french_data[date_column]).dt.date

fama_french_data.head()

,date,Mkt-RF,SMB,HML,RMW,CMA,RF,WML
0,1990-07-31,0.0446,0.0020,-0.0152,0.0028,0.0110,0.0068,NaN
1,1990-08-31,-0.1088,0.0025,-0.0030,-0.0050,0.0166,0.0066,NaN
2,1990-09-30,-0.1219,0.0192,0.0044,0.0033,0.0178,0.0060,NaN
3,1990-10-31,0.0645,-0.0258,-0.0112,0.0086,-0.0073,0.0068,NaN
4,1990-11-30,-0.0042,-0.0266,0.0057,0.0025,-0.0033,0.0057,0.0245


### 2.20 Recession Indicator

In [77]:
# Change date column to date only (remove time component)
recession_indicator["date"] = pd.to_datetime(recession_indicator["date"]).dt.date

# Rename column to recession_indicator for clarity
recession_indicator.rename(columns={"peak_excluded": "recession_indicator"}, inplace=True)

# Drop peak_included column as it not considered here
recession_indicator.drop(columns=["peak_included"], inplace=True)

recession_indicator.head()

,date,recession_indicator
0,1970-01-31,0
1,1970-02-28,0
2,1970-03-31,0
3,1970-04-30,0
4,1970-05-31,0


## 3. Export Data as XLSX

In [ ]:
output_path = "data/processed_data/02_parameters.xlsx"

file_dict = {
    "constituents": constituents_file,
    "industry": industry_file,
    "country": country_of_headquarters_file,
    "currency": currency_file,
    "presence_matrix_daily": presence_matrix,
    "stock_returns_quarterly": stock_returns_quarterly,
    "stock_returns_monthly": stock_returns_monthly,
    "index_returns_quarterly": index_returns_quarterly,
    "index_returns_monthly": index_returns_monthly,
    "free_float_mcap_quarterly": free_float_mcap_quarterly,
    "free_float_mcap_monthly": free_float_mcap_monthly,
    "outstanding_mcap_quarterly": outstanding_mcap_quarterly,
    "outstanding_mcap_monthly": outstanding_mcap_monthly,
    "size_quarterly": size_file,
    "value_quarterly": book_to_price_quarterly,
    "betas_quarterly": betas_file_quarterly,
    "profitability_quarterly": profitability_ratio,
    "investment_quarterly": total_assets_growth_rate,
    "price_momentum_quarterly": momentum_quarterly,
    "esg_ff_monthly": esg_ff_monthly,
    "esg_nff_monthly": esg_nff_monthly,
    "esg_ff_quarterly": esg_ff_quarterly,
    "esg_nff_quarterly": esg_nff_quarterly,
    "esg_momentum_monthly": esg_momentum_monthly,
    "esg_momentum_quarterly": esg_momentum_quarterly,
    "dividend_yield_monthly": monthly_dividend_yield,
    "yield_monthly": three_month_yield,
    "term_spread_monthly": term_spread,
    "default_spread_monthly": bond_yields,
    "fama_french_data_monthly": fama_french_data,
    "risk_free_rate_monthly": risk_free_rate_monthly,
    "recession_indicator_monthly": recession_indicator
}

start = pd.Timestamp(constituent_cut_off_date).date()
end = pd.Timestamp(end_date).date()

with pd.ExcelWriter(output_path) as writer:
    for sheet_name, file in file_dict.items():

        if date_column in file.columns:
            file = file[(file[date_column] >= start) & (file[date_column] <= end)]

        file.to_excel(writer, sheet_name=sheet_name, index = False)
        print(f"Written sheet: {sheet_name}")

print(f"All data successfully written to {output_path}")

Written sheet: constituents
Written sheet: industry
Written sheet: country
Written sheet: currency
Written sheet: presence_matrix_daily
Written sheet: stock_returns_quarterly
Written sheet: stock_returns_monthly
Written sheet: index_returns_quarterly
Written sheet: index_returns_monthly
Written sheet: free_float_mcap_quarterly
Written sheet: free_float_mcap_monthly
Written sheet: outstanding_mcap_quarterly
Written sheet: outstanding_mcap_monthly
Written sheet: size_quarterly
Written sheet: value_quarterly
Written sheet: betas_quarterly
Written sheet: profitability_quarterly
Written sheet: investment_quarterly
Written sheet: price_momentum_quarterly
Written sheet: esg_ff_monthly
Written sheet: esg_nff_monthly
Written sheet: esg_ff_quarterly
Written sheet: esg_nff_quarterly
Written sheet: esg_momentum_monthly
Written sheet: esg_momentum_quarterly
Written sheet: dividend_yield_monthly
Written sheet: yield_monthly
Written sheet: term_spread_monthly
Written sheet: default_spread_monthly
Wri